<a href="https://colab.research.google.com/github/jaypra649/ai-disease-risk-prediction/blob/main/diabetes_risk_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# STEP 1: Import the required libraries

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import joblib


# STEP 2: Load the public dataset

url = (
    "https://raw.githubusercontent.com/"
    "npradaschnor/Pima-Indians-Diabetes-Dataset/"
    "master/diabetes.csv"
)

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nFirst five rows:")
print(df.head())

print("\nTarget distribution:")
print(df["Outcome"].value_counts())


# STEP 3: Separate input features and target

X = df.drop(columns=["Outcome"]).copy()
y = df["Outcome"].copy()


# STEP 4: Handle invalid zero measurements
# Zero is not a valid measurement for these fields
# in this dataset, so treat it as missing.
# Zero pregnancies, however, is a valid value.

zero_as_missing = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

X[zero_as_missing] = X[zero_as_missing].replace(0, float("nan"))


# STEP 5: Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))


# STEP 6: Build a preprocessing and model pipeline

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])


# STEP 7: Train the model

model.fit(X_train, y_train)

print("\nModel training completed!")


# STEP 8: Make predictions on unseen test records

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


# STEP 9: Evaluate the model

print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 3))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Outcome 0", "Outcome 1"],
        zero_division=0
    )
)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print(
    "\nROC-AUC:",
    round(roc_auc_score(y_test, y_prob), 3)
)


# STEP 10: Save the complete trained pipeline

joblib.dump(model, "diabetes_model.joblib")

print("\nSaved model to diabetes_model.joblib")


Dataset loaded successfully!
Dataset shape: (768, 9)

First five rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Target distribution:
Outcome
0    500
1    268
Name: count, dtype: int64

Training records: 614
Testing records: 154

Model training completed!

Accuracy: 0.708

Classification report:
              precision    recall  f1-score   

# New Section

In [3]:

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    accuracy_score
)

# y_prob was generated by the trained model in Step 1.
# It contains estimated probabilities for Outcome 1.

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

print("Threshold comparison")
print("-" * 75)
print(
    f"{'Threshold':<12}"
    f"{'Accuracy':<12}"
    f"{'Precision':<12}"
    f"{'Recall':<12}"
)

for threshold in thresholds:
    # Classify as Outcome 1 at or above the threshold.
    y_threshold = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_test, y_threshold)
    prec = precision_score(
        y_test, y_threshold, zero_division=0
    )
    rec = recall_score(
        y_test, y_threshold, zero_division=0
    )

    print(
        f"{threshold:<12.2f}"
        f"{acc:<12.3f}"
        f"{prec:<12.3f}"
        f"{rec:<12.3f}"
    )


Threshold comparison
---------------------------------------------------------------------------
Threshold   Accuracy    Precision   Recall      
0.30        0.740       0.597       0.796       
0.35        0.721       0.587       0.685       
0.40        0.734       0.614       0.648       
0.45        0.708       0.588       0.556       
0.50        0.708       0.600       0.500       


# New Section

In [2]:
from google.colab import files

files.download("diabetes_model.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# New Section

In [4]:

import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# X and y are from your previous notebook cell.
# X already has invalid zero measurements replaced with NaN.

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ]),

    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DecisionTreeClassifier(
            max_depth=4,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            random_state=42
        ))
    ])
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "roc_auc": "roc_auc"
        },
        return_train_score=False
    )

    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })

results_df = pd.DataFrame(results)
print(results_df.round(3).to_string(index=False))


              Model  Accuracy  Precision  Recall  ROC-AUC
Logistic Regression     0.772      0.726   0.571    0.837
      Decision Tree     0.728      0.612   0.638    0.768
      Random Forest     0.764      0.691   0.597    0.836


# New Section

In [5]:

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

# Reuse the Logistic Regression pipeline from earlier.
logistic_model = models["Logistic Regression"]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Each probability is generated out-of-fold.
oof_prob = cross_val_predict(
    logistic_model,
    X,
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]

print("Out-of-fold ROC-AUC:",
      round(roc_auc_score(y, oof_prob), 3))

rows = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50]:
    pred = (oof_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(
            y, pred, zero_division=0
        ),
        "Recall": recall_score(
            y, pred, zero_division=0
        ),
        "Specificity": tn / (tn + fp),
        "False Negatives": fn,
        "False Positives": fp
    })

comparison = pd.DataFrame(rows)

print("\nOut-of-fold threshold comparison:")
print(comparison.round(3).to_string(index=False))


Out-of-fold ROC-AUC: 0.835

Out-of-fold threshold comparison:
 Threshold  Accuracy  Precision  Recall  Specificity  False Negatives  False Positives
      0.30     0.733      0.588   0.787        0.704               57              148
      0.35     0.753      0.625   0.728        0.766               73              117
      0.40     0.759      0.652   0.664        0.810               90               95
      0.45     0.766      0.683   0.612        0.848              104               76
      0.50     0.772      0.718   0.571        0.880              115               60


In [6]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

final_model.fit(X, y)

joblib.dump(final_model, "diabetes_model.joblib")

print("Model trained and saved successfully!")

Model trained and saved successfully!


In [7]:
import pandas as pd

sample = pd.DataFrame([{
    "Pregnancies": 2,
    "Glucose": 120,
    "BloodPressure": 70,
    "SkinThickness": 25,
    "Insulin": 80,
    "BMI": 28.5,
    "DiabetesPedigreeFunction": 0.4,
    "Age": 30
}])

probability = final_model.predict_proba(sample)[0][1]

print(f"Model score: {probability:.2%}")

Model score: 17.56%
